In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
files = os.listdir(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics"
)
files

['sampled_all_systems.csv',
 'concat_flaml.csv',
 'concat_ag.csv',
 'concat_rf.csv',
 'hpo_all_systems.csv',
 'result_equal_monotonic_rf_equal.csv',
 'result_monotonic_rf_equal.csv',
 'result_equal_monotonic_flaml_equal.csv',
 'result_monotonic_flaml_equal.csv',
 'result_monotonic_ag_equal.csv',
 'result_monotonic_llama.csv',
 'result_monotonic_mistral.csv',
 'llama.csv',
 'mistral.csv',
 'result_equal_monotonic_ag_equal.csv']

# FLAML

In [ ]:
system = "flaml"
system_map = {
    "rf": "Random Forest with Random Search",
    "ag": "AutoGluon",
    "flaml": "FLAML",
}

## HPO

In [ ]:
hpo = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics/hpo_all_systems.csv",
    sep=";"
)
hpo = hpo[hpo["automl_system"] == system_map[system]]
hpo["are_concat_pred_equal"] = hpo.apply(
    lambda row: row["trained_dataset"] == row["predicted_dataset"],
    axis=1
)
equal_hpo = hpo[hpo["are_concat_pred_equal"]]
diff_hpo = hpo[~(hpo["are_concat_pred_equal"])]

In [ ]:
# Looking at one HPO equal dataset example
equal_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
235,ua_01,ua_01,FLAML,gmm_random,100,[],[],False,0.55838,0.627626,"['3', '10', '1', '7', '5']","['3', '10', '1', '7', '5']",True


In [ ]:
# Looking at diff HPO equal dataset example
diff_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
231,ua_01,ua_03,FLAML,gmm_random,100,['3'],[],False,0.666016,0.660813,"['10', '3', '1', '7', '5']","['10', '7', '1', '5']",False
232,ua_01,ua_04,FLAML,gmm_random,100,[],[],False,0.200040,0.263953,"['10', '3', '1', '7', '5']","['3', '10', '1', '7', '5']",False
233,ua_01,ua_05,FLAML,gmm_random,100,['3'],[],False,0.776119,0.721393,"['3', '10', '1', '7', '5']","['10', '7', '1', '5']",False
234,ua_01,ua_06,FLAML,gmm_random,100,['3'],[],False,0.442095,0.460800,"['3', '10', '1', '7', '5']","['10', '7', '1', '5']",False


## Concat

In [ ]:
automl_concat_df = pd.read_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/concat_{system}.csv",
    sep=";"
)

adapt_mask = automl_concat_df["concatenated_dataset"] == automl_concat_df["predicted_dataset"]
aug_mask = automl_concat_df["trained_dataset"] == automl_concat_df["predicted_dataset"]

equal_concat_df = automl_concat_df.loc[adapt_mask & (~aug_mask)]
diff_concat_df = automl_concat_df.loc[(~adapt_mask) & (~aug_mask)]

In [ ]:
# Looking at one equal concat dataset example
equal_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
720,ua_01,ua_03,ua_03,FLAML,gmm_random,25,"['3', '5', '7']",[],False,0.666016,0.665366,"['3', '5', '10', '1', '7']","['10', '1']"
725,ua_01,ua_03,ua_03,FLAML,gmm_random,50,['3'],[],False,0.666016,0.668293,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
730,ua_01,ua_03,ua_03,FLAML,gmm_random,75,"['3', '5']",[],False,0.666016,0.667317,"['3', '5', '10', '1', '7']","['1', '10', '7']"
736,ua_01,ua_04,ua_04,FLAML,gmm_random,25,[],[],False,0.200040,0.411882,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
741,ua_01,ua_04,ua_04,FLAML,gmm_random,50,[],[],False,0.200040,0.412482,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
746,ua_01,ua_04,ua_04,FLAML,gmm_random,75,[],[],False,0.200040,0.436487,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
752,ua_01,ua_05,ua_05,FLAML,gmm_random,25,[],[],False,0.776119,0.699005,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
757,ua_01,ua_05,ua_05,FLAML,gmm_random,50,['3'],[],False,0.776119,0.758706,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
762,ua_01,ua_05,ua_05,FLAML,gmm_random,75,[],[],False,0.776119,0.708955,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
768,ua_01,ua_06,ua_06,FLAML,gmm_random,25,[],[],False,0.442095,0.509922,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"


In [ ]:
# Looking at one diff concat dataset example
diff_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
721,ua_01,ua_04,ua_03,FLAML,gmm_random,25,"['3', '5', '7']",[],False,0.200040,0.235647,"['3', '5', '10', '1', '7']","['10', '1']"
722,ua_01,ua_05,ua_03,FLAML,gmm_random,25,"['3', '5', '7']",[],False,0.776119,0.761194,"['3', '5', '10', '1', '7']","['10', '1']"
723,ua_01,ua_06,ua_03,FLAML,gmm_random,25,"['3', '5', '7']",[],False,0.442095,0.450553,"['3', '5', '10', '1', '7']","['10', '1']"
726,ua_01,ua_04,ua_03,FLAML,gmm_random,50,[],[],False,0.200040,0.255051,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
727,ua_01,ua_05,ua_03,FLAML,gmm_random,50,['3'],[],False,0.776119,0.743781,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
728,ua_01,ua_06,ua_03,FLAML,gmm_random,50,['3'],[],False,0.442095,0.456571,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
731,ua_01,ua_04,ua_03,FLAML,gmm_random,75,['3'],[],False,0.200040,0.242048,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
732,ua_01,ua_05,ua_03,FLAML,gmm_random,75,"['3', '7']",[],False,0.776119,0.756219,"['3', '5', '10', '1', '7']","['5', '10', '1']"
733,ua_01,ua_06,ua_03,FLAML,gmm_random,75,"['3', '5']",[],False,0.442095,0.453318,"['3', '5', '10', '1', '7']","['7', '10', '1']"
735,ua_01,ua_03,ua_04,FLAML,gmm_random,25,[],[],False,0.666016,0.498862,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"


## Equal

### 25, 50, 75

In [ ]:
grouped_equal_concat_df = (
    equal_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_equal_concat_df = grouped_equal_concat_df.reset_index()
grouped_equal_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,FLAML,gmm_random,25,0.000000
1,ed_01,FLAML,gmm_random,50,0.029046
2,ed_01,FLAML,gmm_random,75,0.045643
3,ed_02,FLAML,gmm_random,25,0.553659
4,ed_02,FLAML,gmm_random,50,0.557678
...,...,...,...,...,...
100,ua_05,FLAML,gmm_random,50,0.753234
101,ua_05,FLAML,gmm_random,75,0.747761
102,ua_06,FLAML,gmm_random,25,0.518575
103,ua_06,FLAML,gmm_random,50,0.521113


### 0%

In [ ]:
grouped_zero_pct_hpo = (
    diff_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,FLAML,gmm_random,0.000000,0
1,ed_02,FLAML,gmm_random,0.031118,0
2,fn_02,FLAML,gmm_random,0.352778,0
3,fn_05,FLAML,gmm_random,0.387069,0
4,fn_06,FLAML,gmm_random,0.321519,0
5,fn_07,FLAML,gmm_random,0.394886,0
6,fn_09,FLAML,gmm_random,0.268231,0
7,fn_10,FLAML,gmm_random,0.291667,0
8,hs_01,FLAML,gmm_random,0.435168,0
9,hs_02,FLAML,gmm_random,0.241483,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,FLAML,gmm_random,0.439834,100
1,ed_02,FLAML,gmm_random,0.553599,100
2,ed_03,FLAML,gmm_random,0.381555,100
3,ed_04,FLAML,gmm_random,0.915740,100
4,ed_05,FLAML,gmm_random,0.391667,100
5,fn_01,FLAML,gmm_random,0.236852,100
6,fn_02,FLAML,gmm_random,0.541667,100
7,fn_03,FLAML,gmm_random,0.225308,100
8,fn_04,FLAML,gmm_random,0.563565,100
9,fn_05,FLAML,gmm_random,0.470249,100


### Result

In [ ]:
result_equal = (
    pd.concat(
        [
            grouped_equal_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
            # Correct is:
            #   grouped_equal_concat_df,
            #   grouped_100_pct_hpo,
        ],
        axis=0
    )
    .sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
)
monotonic_equal = (
    result_equal.sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False,
        observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_equal = (
    result_equal.merge(
        monotonic_equal,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_equal["predicted_dataset"].value_counts()

assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert ~grouped_equal_concat_df["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in concatenated datasets"
assert ~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_equal = result_equal[~result_equal["predicted_dataset"].isin(isolated)]
result_equal.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_equal_monotonic_{system}_equal.csv",
    sep=";"
)
result_equal

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,FLAML,gmm_random,0,0.000000,True
1,ed_01,FLAML,gmm_random,25,0.000000,True
2,ed_01,FLAML,gmm_random,50,0.029046,True
3,ed_01,FLAML,gmm_random,75,0.045643,True
4,ed_01,FLAML,gmm_random,100,0.439834,True
...,...,...,...,...,...,...
184,ua_06,FLAML,gmm_random,0,0.437183,True
185,ua_06,FLAML,gmm_random,25,0.518575,True
186,ua_06,FLAML,gmm_random,50,0.521113,True
187,ua_06,FLAML,gmm_random,75,0.521666,True


## Diff

### 25, 50, 75

In [ ]:
grouped_diff_concat_df = (
    diff_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_diff_concat_df = grouped_diff_concat_df.reset_index()
grouped_diff_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,FLAML,gmm_random,25,0.004149
1,ed_01,FLAML,gmm_random,50,0.024896
2,ed_01,FLAML,gmm_random,75,0.062241
3,ed_02,FLAML,gmm_random,25,0.006208
4,ed_02,FLAML,gmm_random,50,0.018017
...,...,...,...,...,...
70,ua_05,FLAML,gmm_random,50,0.603078
71,ua_05,FLAML,gmm_random,75,0.602612
72,ua_06,FLAML,gmm_random,25,0.442613
73,ua_06,FLAML,gmm_random,50,0.448987


### 0%

In [ ]:
grouped_zero_pct_hpo = (
      diff_hpo.groupby(
          [
              "predicted_dataset",
              "automl_system",
              "method"
          ]
      )
      .agg(
          {
              "hold_out_f1_score": "mean"
          }
      )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,FLAML,gmm_random,0.000000,0
1,ed_02,FLAML,gmm_random,0.031118,0
2,fn_02,FLAML,gmm_random,0.352778,0
3,fn_05,FLAML,gmm_random,0.387069,0
4,fn_06,FLAML,gmm_random,0.321519,0
5,fn_07,FLAML,gmm_random,0.394886,0
6,fn_09,FLAML,gmm_random,0.268231,0
7,fn_10,FLAML,gmm_random,0.291667,0
8,hs_01,FLAML,gmm_random,0.435168,0
9,hs_02,FLAML,gmm_random,0.241483,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,FLAML,gmm_random,0.439834,100
1,ed_02,FLAML,gmm_random,0.553599,100
2,ed_03,FLAML,gmm_random,0.381555,100
3,ed_04,FLAML,gmm_random,0.915740,100
4,ed_05,FLAML,gmm_random,0.391667,100
5,fn_01,FLAML,gmm_random,0.236852,100
6,fn_02,FLAML,gmm_random,0.541667,100
7,fn_03,FLAML,gmm_random,0.225308,100
8,fn_04,FLAML,gmm_random,0.563565,100
9,fn_05,FLAML,gmm_random,0.470249,100


### Result

In [ ]:
result_diff = (
    pd.concat(
        [
            grouped_diff_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
        ],
        axis=0
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)
monotonic_diff = (
    result_diff.sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False, observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_diff = (
    result_diff.merge(
        monotonic_diff,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_diff["predicted_dataset"].value_counts()

# assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert (~grouped_equal_concat_df["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in concatenated datasets"
assert (~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_diff = result_diff[~result_diff["predicted_dataset"].isin(isolated)]
result_diff.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_monotonic_{system}_equal.csv",
    sep=";"
)
result_diff

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,FLAML,gmm_random,0,0.000000,True
1,ed_01,FLAML,gmm_random,25,0.004149,True
2,ed_01,FLAML,gmm_random,50,0.024896,True
3,ed_01,FLAML,gmm_random,75,0.062241,True
4,ed_01,FLAML,gmm_random,100,0.439834,True
...,...,...,...,...,...,...
154,ua_06,FLAML,gmm_random,0,0.437183,False
155,ua_06,FLAML,gmm_random,25,0.442613,False
156,ua_06,FLAML,gmm_random,50,0.448987,False
157,ua_06,FLAML,gmm_random,75,0.445917,False


# Analysis Examples

Some examples of interesting analysis below:


## Monotonic Analysis

### Equal count

In [ ]:
is_monotonic_increasing_equal = result_equal["is_monotonic_increasing"]

monotonic_increasing_equal = result_equal.loc[is_monotonic_increasing_equal]
non_monotonic_increasing_equal = result_equal.loc[~is_monotonic_increasing_equal]

print(f"Number EQUAL Monotonic increasing: {len(monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number EQUAL Non-monotonic increasing: {len(non_monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

Number EQUAL Monotonic increasing: 14
['ed_01', 'fn_09', 'fn_10', 'hs_02', 'hs_03', 'hs_08', 'hs_09', 'hs_11', 'pc_01', 'sd_03', 'sd_04', 'ua_01', 'ua_03', 'ua_06']
Number EQUAL Non-monotonic increasing: 21
['ed_02', 'fn_02', 'fn_05', 'fn_06', 'fn_07', 'hs_01', 'hs_04', 'hs_06', 'hs_10', 'hs_12', 'hs_13', 'hs_14', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_04', 'ua_05']


### Diff count

In [ ]:
is_monotonic_increasing_diff = result_diff["is_monotonic_increasing"]

monotonic_increasing_diff = result_diff.loc[is_monotonic_increasing_diff]
non_monotonic_increasing_diff = result_diff.loc[~is_monotonic_increasing_diff]

print(f"Number diff Monotonic increasing: {len(monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number diff Non-monotonic increasing: {len(non_monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

Number diff Monotonic increasing: 18
['ed_01', 'fn_05', 'fn_06', 'fn_07', 'fn_09', 'fn_10', 'hs_01', 'hs_04', 'hs_08', 'hs_11', 'hs_14', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'sd_03', 'sd_04', 'ua_04']
Number diff Non-monotonic increasing: 17
['ed_02', 'fn_02', 'hs_02', 'hs_03', 'hs_06', 'hs_09', 'hs_10', 'hs_12', 'hs_13', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_01', 'ua_03', 'ua_05', 'ua_06']


### Non-monotonicic equal f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_equal,
            (
                non_monotonic_increasing_equal[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
6,ed_02,FLAML,gmm_random,25,0.553659,0.522541
7,ed_02,FLAML,gmm_random,50,0.557678,0.004019
8,ed_02,FLAML,gmm_random,75,0.521079,-0.036599
9,ed_02,FLAML,gmm_random,100,0.553599,0.032520
15,fn_02,FLAML,gmm_random,25,0.481944,0.129167
...,...,...,...,...,...,...
178,ua_04,FLAML,gmm_random,100,0.450990,-0.002260
180,ua_05,FLAML,gmm_random,25,0.716418,0.084577
181,ua_05,FLAML,gmm_random,50,0.753234,0.036816
182,ua_05,FLAML,gmm_random,75,0.747761,-0.005473


### Non-monotonic diff f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_diff,
            (
                non_monotonic_increasing_diff[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
6,ed_02,FLAML,gmm_random,25,0.006208,-0.024910
7,ed_02,FLAML,gmm_random,50,0.018017,0.011809
8,ed_02,FLAML,gmm_random,75,0.014674,-0.003344
9,ed_02,FLAML,gmm_random,100,0.553599,0.538925
15,fn_02,FLAML,gmm_random,25,0.406250,0.053472
...,...,...,...,...,...,...
153,ua_05,FLAML,gmm_random,100,0.781095,0.178483
155,ua_06,FLAML,gmm_random,25,0.442613,0.005431
156,ua_06,FLAML,gmm_random,50,0.448987,0.006374
157,ua_06,FLAML,gmm_random,75,0.445917,-0.003070


In [ ]:
da_analysis = result_diff[result_diff["percentage"].isin([0, 100])] # or result_equal since it has the same effect
da_analysis = da_analysis.merge(da_analysis, on=["predicted_dataset", "automl_system", "method"])
da_analysis = da_analysis[
    (da_analysis["percentage_x"] != da_analysis["percentage_y"])
]
da_analysis = da_analysis.drop_duplicates(["predicted_dataset", "automl_system", "method"])
da_analysis["diff"] = da_analysis["hold_out_f1_score_y"] - da_analysis["hold_out_f1_score_x"]
da_analysis.sort_values("diff", ignore_index=True)

,predicted_dataset,automl_system,method,percentage_x,hold_out_f1_score_x,is_monotonic_increasing_x,percentage_y,hold_out_f1_score_y,is_monotonic_increasing_y,diff
0,hs_06,FLAML,gmm_random,0,0.575580,False,100,0.574887,False,-0.000693
1,hs_10,FLAML,gmm_random,0,0.595833,False,100,0.630000,False,0.034167
2,hs_04,FLAML,gmm_random,0,0.883704,True,100,0.921104,True,0.037400
3,fn_07,FLAML,gmm_random,0,0.394886,True,100,0.471591,True,0.076705
4,fn_05,FLAML,gmm_random,0,0.387069,True,100,0.470249,True,0.083180
5,hs_09,FLAML,gmm_random,0,0.862682,False,100,0.948304,False,0.085622
6,ua_03,FLAML,gmm_random,0,0.578797,False,100,0.666341,False,0.087545
7,ua_01,FLAML,gmm_random,0,0.539898,False,100,0.627626,False,0.087728
8,ua_06,FLAML,gmm_random,0,0.437183,False,100,0.538874,False,0.101692
9,pc_08,FLAML,gmm_random,0,0.366452,False,100,0.483202,False,0.116750
